# Confirmatory Factor Analysis (CFA) Preprocessing

Given $M$ models, $R$ repetitions, and $N$ construct items, preprocessing produces a score matrix $(M * R) \times N$ for each construct-approach pair, where each row corresponds to a single survey instance (one model responds to $N$ items sequentially), each column corresponds to a specific construct item, and each cell contains the respective agreement score.
Note that refusals were replaced by mean scores of repetitions for the corresponding item of the same model.
In case a model refused for each repetition given some item, we imputed the neutral threshold score of the corresponding construct.

In [20]:
import json
import numpy as np
import pandas as pd
from llm_audit import BASE_DIR
from llm_audit.datasets.util import get_dataset_by_label

DATASET_LABELS: list[str] = [
    "F",
    "LAS",
    "D",
    "A",
    "AA",
    "RWA",
    "RWA3D",
    "KSA3",
    "ACT",
    "VSA",
    "ASC",
    "APC",
    "CSM",
    "DW",
    "BDW",
]
OLMO_ORDER = {
    "Olmo 3.1 32B": "Olmo00",
    "Olmo3 7B Base": "Olmo01",
    "Olmo3 7B Instruct SFT": "Olmo02",
    "Olmo3 7B Instruct DPO": "Olmo03",
    "Olmo3 7B Instruct RLVR": "Olmo04",
}
LANGUAGE = "en"

with open(BASE_DIR / "resources" / "input" / "models" / "final_complete.json") as f:
    models = json.load(f)

models.sort(key=lambda m: (m["group"], OLMO_ORDER.get(m["name_short"], m["name_short"])))
model_labels = [m["name"] for m in models]


def impute_scores(scores: pd.Series, neutral_threshold: float) -> list[float]:
    """
    Impute missing values in a repetition score series for one (model, item) pair.
    - All missing -> neutral threshold for all repetitions.
    - Some missing -> per-item mean over non-missing repetitions.
    """
    if scores.isna().all():
        return [neutral_threshold] * len(scores)
    mean_value = round(scores.mean(), 4)
    return scores.fillna(mean_value).tolist()


# Load and pre-filter data once
raw = pd.read_csv(BASE_DIR / "eval" / "data" / "tidy" / "construct_scores_ensemble.csv")
raw_default = raw[(raw["language"] == LANGUAGE) & (raw["experiment_ablation"] == "default")].copy()

# Pre-index for fast lookups: (dataset, experiment_type, model, statement_id) -> scores
grouped = raw_default.groupby(["dataset", "experiment_type", "model", "statement_id"])["score"]

for dataset_label in DATASET_LABELS:
    for experiment_type_label in ["closed_question", "open_question"]:
        dataset = get_dataset_by_label(dataset_label=dataset_label)
        ids: list[int] = list(dataset.get_ids(language=LANGUAGE))
        neutral_threshold = dataset.get_agreement_discriminator_threshold()

        header = np.array([f"X{id}" for id in ids])
        model_matrices: list[np.ndarray] = []

        for model_label in model_labels:
            item_vectors: list[list[float]] = []

            for id in ids:
                key = (dataset_label, experiment_type_label, model_label, id)
                if key not in grouped.groups:
                    raise ValueError(
                        f"No data for dataset={dataset_label}, "
                        f"experiment={experiment_type_label}, "
                        f"model={model_label}, item={id}."
                    )
                scores = grouped.get_group(key).reset_index(drop=True)
                item_vectors.append(impute_scores(scores, neutral_threshold))

            # Shape: (n_repetitions, n_items) → transpose to (n_items, n_repetitions)
            model_matrices.append(np.array(item_vectors, dtype=float).T)

        # Stack header + all model repetition rows: shape (1 + M*R, C)
        array_stack = np.vstack([header, *model_matrices])

        output_file = BASE_DIR / "eval" / "data" / "cfa" / f"{dataset_label}_{LANGUAGE}_{experiment_type_label}.csv"
        output_file.parent.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(array_stack[1:], columns=array_stack[0]).to_csv(output_file, index=False)